# 04.3.1 - Supervised Learning: Ekstraksi Aturan Bisnis Decision Tree (K-Means DE)

## 1. Tujuan Analisis
Ini adalah tahap pengujian untuk dataset ketiga dan terakhir kita: hasil pelabelan dari algoritma **K-Means DE**.

Kita kembali menggunakan algoritma *Decision Tree* (Pohon Keputusan). Ingat, tujuan utama *Decision Tree* di sini bukan semata-mata mencari akurasi tertinggi, melainkan memanfaatkan sifatnya yang transparan (*White-Box*). Kita ingin mengekstrak **Aturan Bisnis (Business Rules)** dari kelompok pelanggan hasil pelabelan algoritma DE agar bisa dibaca dan digunakan oleh tim bisnis atau *marketing*.

In [1]:
import pandas as pd
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report

# 1. BACA DATA (Gunakan Pemenang Clustering: DE)
df = pd.read_csv('../data/Labeled/hasildata_kmeans-de.csv')

# Pisahkan Fitur (Var1 - Var11) dan Target (Cluster hasil DE)
fitur = [f'Var{i}' for i in range(1, 12)]
X = df[fitur]
y = df['Cluster']

# Splitting Data: 80% untuk Train (belajar), 20% untuk Test (ujian)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Training Model Decision Tree
Kita tetap mengunci batas kedalaman maksimal model di angka 4 (`max_depth=4`). Tujuannya agar pohon keputusan tidak membuat cabang kondisi yang terlalu panjang dan rumit. Kita menginginkan aturan bisnis yang sederhana dan siap pakai di lapangan.

In [2]:
# 2. TRAINING MODEL DECISION TREE
model_dt = DecisionTreeClassifier(random_state=42, max_depth=4)
model_dt.fit(X_train, y_train)

# Minta model menebak data ujian (Test)
prediksi_dt = model_dt.predict(X_test)

# Tampilkan Rapor Performa
print("=== CLASSIFICATION REPORT: DECISION TREE (DE) ===\n")
print(classification_report(y_test, prediksi_dt))

=== CLASSIFICATION REPORT: DECISION TREE (DE) ===

              precision    recall  f1-score   support

           0       0.94      0.89      0.91       138
           1       0.87      0.75      0.81       165
           2       0.95      0.90      0.93       165
           3       0.96      0.88      0.92        84
           4       0.85      0.71      0.77       111
           5       0.70      0.91      0.79       204

    accuracy                           0.85       867
   macro avg       0.88      0.84      0.86       867
weighted avg       0.86      0.85      0.85       867



### Analisis Performa (DE vs Standard & QLDE)
Hasil rapor di atas menunjukkan bahwa akurasi *Decision Tree* pada dataset DE mencapai **85%** (atau `0.85`). 

Angka ini merupakan yang **paling tinggi** jika dibandingkan dengan hasil *Decision Tree* pada dataset Standard (83%) dan QLDE (83%). Ini mengindikasikan bahwa kelompok (*cluster*) pelanggan yang dibentuk oleh metode DE memiliki batasan pola yang lebih rapi dan linier, sehingga lebih mudah dipisahkan oleh kondisi If-Else sederhana.

## 3. Ekstraksi Aturan Bisnis DE
Sekarang, mari kita cetak teks aturan bisnis yang berhasil diekstrak oleh model dari data DE ini.

In [3]:
# 3. EKSTRAKSI ATURAN BISNIS
aturan_bisnis = export_text(model_dt, feature_names=fitur)
print("=== BUSINESS RULES DE (ATURAN LOGIKA PELANGGAN) ===\n")
print(aturan_bisnis)

=== BUSINESS RULES DE (ATURAN LOGIKA PELANGGAN) ===

|--- Var7 <= 7.35
|   |--- Var1 <= 105.50
|   |   |--- Var9 <= 0.50
|   |   |   |--- Var11 <= 152.01
|   |   |   |   |--- class: 0
|   |   |   |--- Var11 >  152.01
|   |   |   |   |--- class: 3
|   |   |--- Var9 >  0.50
|   |   |   |--- Var4 <= 2195.45
|   |   |   |   |--- class: 0
|   |   |   |--- Var4 >  2195.45
|   |   |   |   |--- class: 4
|   |--- Var1 >  105.50
|   |   |--- Var9 <= 0.50
|   |   |   |--- Var11 <= 232.38
|   |   |   |   |--- class: 2
|   |   |   |--- Var11 >  232.38
|   |   |   |   |--- class: 3
|   |   |--- Var9 >  0.50
|   |   |   |--- Var8 <= 127.50
|   |   |   |   |--- class: 2
|   |   |   |--- Var8 >  127.50
|   |   |   |   |--- class: 2
|--- Var7 >  7.35
|   |--- Var8 <= 136.83
|   |   |--- Var4 <= 3541.32
|   |   |   |--- Var9 <= 0.50
|   |   |   |   |--- class: 3
|   |   |   |--- Var9 >  0.50
|   |   |   |   |--- class: 5
|   |   |--- Var4 >  3541.32
|   |   |   |--- Var10 <= 0.50
|   |   |   |   |--- cla

### Cara Membaca Aturan di Atas:
Bagan ini adalah panduan klasifikasi pelanggan versi algoritma DE.
Baca dari atas ke bawah. Misalnya: Jika **Var7 <= 7.35**, dan **Var1 <= 105.50**, lalu **Var9 <= 0.50**, dan nilai **Var11 <= 152.01**, maka pelanggan tersebut masuk ke `class: 0` (Cluster 0).

## 4. Menyimpan Model
Terakhir, kita simpan model *Decision Tree* ini dengan nama khusus `_de` agar bisa dipanggil kembali oleh sistem tanpa tertukar.

In [4]:
# 4. EXPORT MODEL KE FOLDER 'models'
os.makedirs('../models', exist_ok=True)

# Simpan model dengan akhiran _de
joblib.dump(model_dt, '../models/model_dt_classification_de.pkl')
print("\n[SUCCESS] Model Decision Tree berhasil diekspor ke '../models/model_dt_classification_de.pkl'")


[SUCCESS] Model Decision Tree berhasil diekspor ke '../models/model_dt_classification_de.pkl'
